In [9]:
import torch
import torchaudio
import soundfile as sf
import numpy as np
import librosa
from pydub import AudioSegment

# 1. Load data

In [10]:
bundle = torchaudio.pipelines.WAV2VEC2_ASR_BASE_960H
model = bundle.get_model()

audio_file = "sample_audio.wav"
audio, sr = librosa.load(audio_file, sr=None)
clip = AudioSegment.from_wav(audio_file)

# 2. Export Function

In [11]:
def export_pydub(audio):
    while True:
        file_name = input("input file name: ")
        if file_name.endswith(".wav"):
            break
        else:
            print("file must ends with .wav")
    audio.export(file_name, format="wav")
    print("Audio saved as: ", file_name)


def export_librosa(audio):
    while True:
        file_name = input("input file name: ")
        if file_name.endswith(".wav"):
            break
        else:
            print("file must ends with .wav")
    sf.write(file_name, np.clip(audio, -1, 1), sr)
    print("Audio saved as: ", file_name)

# 3. Predict Text Wav2Vec

In [12]:
def predict_text(audio_file):
    audio_np, _ = librosa.load(audio_file, sr=bundle.sample_rate, mono=True)
    waveform = torch.from_numpy(audio_np).unsqueeze(0)

    with torch.inference_mode():
        emission, _ = model(waveform)

    class GreedyCTCDecoder(torch.nn.Module):
        def __init__(self, labels):
            super().__init__()
            self.labels = labels
            self.blank = 0

        def forward(self, emission: torch.Tensor)->str:
            indices = torch.argmax(emission, dim=-1)
            indices = torch.unique_consecutive(indices, dim=-1)
            indices = [i for i in indices if i != self.blank]
            return "".join([self.labels[i] for i in indices])

    decoder = GreedyCTCDecoder(bundle.get_labels())
    transcript = decoder(emission[0])
    transcript = transcript.replace("|", " ")
    return transcript

# 4. Augmentation

In [ ]:
# 9 augmentasi
# 1. adjvolume
def adjvolume(clip):
    while True:
        nilai = float(input("Input nilai 1 sampai 5: "))
        if not (1 <= nilai <= 5):
            print("Must be between 1 to 5.")
        else:
            break
    export_pydub(clip + nilai)

# 2. repeat audio
def repeat_audio(clip):
    while True:
        nilai = int(input("Input nilai repeat harus lebih dari 0: "))
        if nilai <= 0:
            print("Must be greater than 0.")
        else:
            break
    export_pydub(clip * nilai)

# 3. fade in
def fade_in(clip):
    while True:
        nilai = int(input("Input nilai fade in harus lebih dari 0: "))
        if nilai <= 0:
            print("Must be greater than 0.")
        else:
            break
    export_pydub(clip.fade_in(nilai))

# 4. fade out
def fade_out(clip):
    while True:
        nilai = int(input("Input nilai 1 sampai 5: "))
        if not (1 <= nilai <= 5):
            print("Must be between 1 to 5.")
        else:
            break
    export_pydub(clip.fade_out(nilai))

# 5. reverse audio
def reverse_audio(clip):
    export_pydub(clip.reverse())

# 6. adjpitch
def adjpitch():
    while True:
        nilai = float(input("Input nilai -5 sampai 5: "))
        if not (-5 <= nilai <= 5):
            print("Must be between -5 to 5.")
        else:
            break
    shift_audio = librosa.effects.pitch_shift(audio, sr=sr, n_steps=nilai)
    export_librosa(shift_audio)

# 7. adjspeed
def adjspeed():
    while True:
        nilai = float(input("Input nilai 1 sampai 5: "))
        if not (1 <= nilai <= 5):
            print("Must be between 1 to 5.")
        else:
            break
    speed_audio = librosa.effects.time_stretch(audio, rate=nilai)
    export_librosa(speed_audio)

# 8. white noise
def white_noise():
    while True:
        nilai = float(input("Input nilai 0.1 sampai 0.5: "))
        if not (0.1 <= nilai <= 0.5):
            print("Must be between 0.1 to 0.5.")
        else:
            break
    noise = np.random.normal(0, audio.std(), audio.size)
    export_librosa(audio + noise * nilai)

# 9. random gain
def random_gain():
    nilai = np.random.uniform(0.5, 1.5)
    print(f"Random gain x {nilai:.2f}")
    export_librosa(audio * nilai)

# 5. Menu Aug

In [14]:
def aug_menu():
    while True:
        print("1. Pitch")
        print("2. White Noise")
        print("3. Repeat")
        print("4. Fade In")
        print("5. Back to Main Menu")

        choice = input(">> ")
        if choice == "1":
            adjpitch()
        elif choice == "2":
            white_noise()
        elif choice == "3":
            repeat_audio(clip)
        elif choice == "4":
            fade_in(clip)
        elif choice == "5":
            break
        else:
            print("Invalid input")

# 6. Main Menu

In [15]:
def main_menu():
    while True:
        print("1. Audio Augmentation")
        print("2. Predict Text")
        print("3. Exit")

        choice = input(">> ")
        if choice == "1":
            print("Audio augmentation now")
            aug_menu()
        elif choice == "2":
            transcript = predict_text(audio_file)
            print("Hasil Transcript: ")
            print(transcript)
        elif choice == "3":
            break
        else:
            print("Invalid input")
    print("Main Menu Closed")

main_menu()

1. Audio Augmentation
2. Predict Text
3. Exit
Audio augmentation now
1. Pitch
2. White Noise
3. Repeat
4. Fade In
5. Back to Main Menu
Audio saved as:  fadein.wav
1. Pitch
2. White Noise
3. Repeat
4. Fade In
5. Back to Main Menu
Must be between 0.1 to 0.5.
Audio saved as:  white.wav
1. Pitch
2. White Noise
3. Repeat
4. Fade In
5. Back to Main Menu
1. Audio Augmentation
2. Predict Text
3. Exit
Hasil Transcript: 
THE PUBLIC ARE ENTREATED TO BEAR IN MIND THAT THIRTEEN YEARS HAVE PASSED SINCE IT WAS FINISHED MANY MORE SINCE IT WAS BEGUN AND THAT DURING THAT PERIOD PLACES MANNERS BOOKS AND OPINIONS HAVE UNDERGONE CONSIDERABLE CHANGES 
1. Audio Augmentation
2. Predict Text
3. Exit
Main Menu Closed
